# Stage1 前日带方向成交量压力：来源覆盖预检

假设来自原 Stage1 在下跌后现金偏高的区间。只读原 Qlib 分钟日历、少量 SOXX 分钟字段和原训练时钟；先验来源年限不足即止步，不构造策略或读取评价收益。

In [1]:
import torch
import hashlib,json
from pathlib import Path
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_signed_volume_pressure_preflight_20261022'
plan=json.loads((card/'plan.json').read_text())
assert hashlib.sha256((root/plan['source_plan']).read_bytes()).hexdigest()==plan['source_plan_sha256']
provider=root/plan['source_provider']
assert hashlib.sha256((provider/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['source_provider_manifest_sha256']
source_card=root/'research/minute_information_pit_allocator_20261012'
source_summary=json.loads((source_card/'summary.json').read_text())
assert source_summary['recorder_id']==plan['source_timing_recorder_id'] and source_summary['nas_evidence']['sha256']==plan['source_nas_archive_sha256']
manifest=json.loads((source_card/'verification_manifest.json').read_text())
entry=next(x for x in manifest['entries'] if x['path'].endswith('/artifacts/inputs/timing'))
assert entry['sha256']==plan['source_timing_artifact_sha256']
assert hashlib.sha256((source_card/entry['path']).read_bytes()).hexdigest()==entry['sha256']
qlib.init(provider_uri=str(provider),region='us',kernels=1)
with R.uri_context(source_summary['recorder_uri']):
    rec=R.get_recorder(recorder_id=source_summary['recorder_id'],experiment_id=source_summary['experiment_id'])
    timing=rec.load_object('inputs/timing')
train=pd.DatetimeIndex(timing.index)
train=train[(train>=pd.Timestamp('2011-01-03')) & (train<=pd.Timestamp('2022-12-30'))]
minute=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2022-12-30 23:59:59',freq='1min'))
source_days=pd.DatetimeIndex(minute.normalize().unique())
missing=train.difference(source_days)
probe=D.features(['SOXX'],['$close','$volume','$price_valid'],start_time='2018-10-12',end_time='2018-10-15 16:00:00',freq='1min')
assert not probe.empty and pd.to_numeric(probe['$volume'],errors='coerce').notna().any()
result={'status':'needs_pre_2018_minute_source_history_no_fit' if len(missing) else 'source_calendar_coverage_passed_only',
 'training_signal_dates':len(train),'minute_source_first':str(minute.min()),'minute_source_last_training':str(minute.max()),
 'source_session_dates_in_train':int(train.isin(source_days).sum()),'missing_training_signal_dates':len(missing),
 'missing_first':str(missing.min()) if len(missing) else None,'missing_last':str(missing.max()) if len(missing) else None,
 'sample_native_SOXX_minute_rows':len(probe),'source_provider_manifest_sha256':plan['source_provider_manifest_sha256'],
 'source_timing_recorder_id':plan['source_timing_recorder_id'],'source_nas_archive_sha256':plan['source_nas_archive_sha256'],
 'raw_minute_candidate_values_computed':0,'market_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,
 'next_action':'Do not fit on a zero-filled 2011-2018 span or select 2018+ as a rescue window. Qualify an original full-training-history minute source before revisiting this candidate.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({k:result[k] for k in ['status','training_signal_dates','source_session_dates_in_train','missing_training_signal_dates','minute_source_first','sample_native_SOXX_minute_rows']})

{'status': 'needs_pre_2018_minute_source_history_no_fit', 'training_signal_dates': 3020, 'source_session_dates_in_train': 1062, 'missing_training_signal_dates': 1958, 'minute_source_first': '2018-10-12 09:31:00', 'sample_native_SOXX_minute_rows': 780}


## 中文结论

Qlib 原生分钟源始于 2018-10-12，原 Stage1 3020 个训练信号日仅 1062 日有分钟源，缺 1958 日至 2018-10-11。Futu 单次只读请求 2011-01-03 返回成功码、0 行；这不是整个历史权限的否定。该方向成交量压力只是一条来源假设，本轮未计算因子、拟合或账户；不填零或缩短训练窗口补救。决定：`needs_source_history_no_fit`。
